# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaryumAkram16/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: The Freshness Multiplier (Finding #4) — the `361+` freshness bucket, 283:1 growth ratio.**

The paper itself flags this as unstable: "its 283:1 ratio is unstable: 283 growing pages versus only 1 declining." My methodology question: what minimum-n threshold is being applied before a ratio is allowed onto the headline chart at all? The paper states a general minimum sample size of "50 per bucket (30 for cross-correlations)" in the ML appendix, but this bucket appears to have far fewer than 50 total observations if only 1 page is declining. Is the 283:1 figure from a bucket that fails the paper's own stated minimum-n rule, and if so, should it appear as a chart bar (even with a caveat) or be excluded from visualization entirely, the same way weak internal constructs were "demoted or removed" elsewhere per the paper's own stated evidence standard?

**Finding 2: ML Appendix — "What Predicts Health?" (Random Forest feature importance).**

The paper is admirably upfront that "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal" — and indeed, Health Score is explicitly defined as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). Average Position shows 43% importance, Impressions 32%, Scroll Depth 15%, CTR 8% — which sums to 98% of total importance concentrated in the exact four components the label is built from. My methodology question: if the label's own ingredients were excluded from the feature set, would there be any meaningful predictive signal left at all, or does this experiment mostly demonstrate that the model can reconstruct a known formula? A version of this analysis with the four label components removed would show whether there's a genuine external predictor (e.g. content age, word count) worth highlighting, versus a result that mainly confirms the arithmetic of Health Score itself.

In [5]:
# No query needed here — this section is a close reading of the paper's own
# disclosed numbers and methodology notes, cited directly above.
print("Methodology questions above reference the paper's own stated evidence "
      "standard (min n=50/bucket) and its own health score formula "
      "(impressions 30 + position 30 + CTR 20 + scroll depth 20 = 100 pts).")

Methodology questions above reference the paper's own stated evidence standard (min n=50/bucket) and its own health score formula (impressions 30 + position 30 + CTR 20 + scroll depth 20 = 100 pts).


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before/after: naive random split vs. the grouped client-holdout split I actually used in Week 5.**

In Week 5, I used a `GroupShuffleSplit` grouped by `client_id` specifically to prevent pages from the same client leaking between train and test. Here I demonstrate *why* that mattered by re-running the identical Random Forest with a naive random row split instead — the mistake this design was built to avoid — and comparing Precision@50 on both.

**What this proves:** The naive split let 27 clients appear in both train and test, and — as expected — it produced an inflated Precision@50 (0.88) that partly reflects the model memorizing client-specific patterns rather than learning a generalizable rule. The grouped split (0 overlap) produced a lower but honest number (0.76), matching exactly what I reported in Week 5. This is a concrete demonstration of why the split design matters: a higher metric is not automatically a better result if it was earned by leaking information the model wouldn't have at real prediction time.

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

url = "https://raw.githubusercontent.com/MaryumAkram16/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df = df[df["impressions_90d"] >= 500].copy()
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

numeric_features = ["impressions_90d", "avg_position", "ctr", "content_age_days",
                     "days_since_last_update", "word_count"]
categorical_features = ["position_tier"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-scores)
    return np.array(labels)[order][:k].mean()

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

# ---------- BEFORE: naive random split (ignores that pages share clients) ----------
X = df[numeric_features + categorical_features]
y = df["is_declining"]
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_naive = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42))])
rf_naive.fit(X_train_naive, y_train_naive)
naive_scores = rf_naive.predict_proba(X_test_naive)[:, 1]
naive_p50 = precision_at_k(naive_scores, y_test_naive.values, 50)

# Check: how much client overlap exists between the naive train/test sets?
train_clients_naive = set(df.loc[X_train_naive.index, "client_id"])
test_clients_naive = set(df.loc[X_test_naive.index, "client_id"])
naive_overlap = len(train_clients_naive & test_clients_naive)

# ---------- AFTER: grouped client-holdout split (what I actually used in Week 5) ----------
groups = df["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))
train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

X_train_grp = train_df[numeric_features + categorical_features]
y_train_grp = train_df["is_declining"]
X_test_grp = test_df[numeric_features + categorical_features]
y_test_grp = test_df["is_declining"]

rf_grouped = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42))])
rf_grouped.fit(X_train_grp, y_train_grp)
grouped_scores = rf_grouped.predict_proba(X_test_grp)[:, 1]
grouped_p50 = precision_at_k(grouped_scores, y_test_grp.values, 50)

grouped_overlap = len(set(train_df["client_id"]) & set(test_df["client_id"]))

# ---------- Comparison ----------
comparison = pd.DataFrame({
    "Split design": ["Naive random split (BEFORE)", "Grouped client-holdout (AFTER — Week 5's actual design)"],
    "Client overlap (train∩test)": [naive_overlap, grouped_overlap],
    "Precision@50": [naive_p50, grouped_p50]
})
print(comparison.to_string(index=False))

                                           Split design  Client overlap (train∩test)  Precision@50
                            Naive random split (BEFORE)                           27          0.88
Grouped client-holdout (AFTER — Week 5's actual design)                            0          0.76


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Leakage audit on my final feature set** (same hunt as Week 3, re-run on the exact features used in the Week-5 model).

In [7]:
scoring_inputs = numeric_features + categorical_features
label_derived_fields = ["trend_direction", "trend_pct", "is_declining"]

leaked = [col for col in scoring_inputs if col in label_derived_fields]
print(f"Final feature set: {scoring_inputs}")
print(f"Label-derived fields checked against: {label_derived_fields}")
print(f"Leakage found? {leaked if leaked else 'None — clean.'}")

# Second check: does any feature correlate suspiciously highly with the label
# (a common leakage red flag), separate from a real predictive relationship?
correlations = df[numeric_features].corrwith(df["is_declining"]).sort_values(key=abs, ascending=False)
print("\nCorrelation of each feature with the label (checking for suspiciously high values):")
print(correlations)
print("\nNone of these approach 0.9+ (which would suggest a feature is secretly encoding the label).")


Final feature set: ['impressions_90d', 'avg_position', 'ctr', 'content_age_days', 'days_since_last_update', 'word_count', 'position_tier']
Label-derived fields checked against: ['trend_direction', 'trend_pct', 'is_declining']
Leakage found? None — clean.

Correlation of each feature with the label (checking for suspiciously high values):
content_age_days         -0.178815
ctr                      -0.104750
impressions_90d          -0.072432
avg_position             -0.051282
days_since_last_update    0.029813
word_count               -0.000617
dtype: float64

None of these approach 0.9+ (which would suggest a feature is secretly encoding the label).


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest original claim (from Week 5's Section 4 interpretation):**

> "The honest takeaway from this week is that a hand-tuned rule targeting one strong, validated signal (CTR-vs-position) can hold its own against generic off-the-shelf models on a small evaluation set."

**Rewritten in safe language:**

On this specific 6-client, 1,260-row held-out test slice, the hand-tuned baseline rule was **observed** to score Precision@50 = 0.90, compared to 0.76 for the Random Forest — a **directional** result that suggests the baseline's targeted CTR-gap logic may capture the validated signal (Signal Check 2, ML-04) more directly than a general-purpose model trained on a broader, unweighted feature set. This is **decision-support** evidence, not a settled conclusion: the test set is small enough that a different train/test client split could shift the ranking of methods, and the comparison has not been repeated across multiple folds. I would need grouped k-fold cross-validation across more client splits before treating "the baseline beats the model" as a stable finding rather than a single observation.

In [8]:
# No new computation needed — this section is a rewrite of prior prose,
# citing the same numbers already verified in Sections 2 (this notebook) and Week 5.
print("Claim rewrite complete — see markdown above. "
      "Original claim used unqualified language ('can hold its own'); "
      "rewrite adds explicit sample-size caveat and softens to observed/directional framing.")


Claim rewrite complete — see markdown above. Original claim used unqualified language ('can hold its own'); rewrite adds explicit sample-size caveat and softens to observed/directional framing.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.